<a href="https://colab.research.google.com/github/coywil26/DIMPLE/blob/master/DIMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DIMPLE — Deep Indel Missense Programmable Library Engineering

Design oligonucleotide libraries for deep mutational scanning. Run the cells in order, filling in the form fields on the right.

> This notebook is the source of truth and lives on `master`. The `colab` branch is deprecated.

In [ ]:
#@title 1. Install

#@markdown Run this once. Clones DIMPLE and installs it with `uv` (fast).

!git clone https://github.com/coywil26/DIMPLE
%cd DIMPLE
!pip install --quiet uv
!uv pip install --system --quiet -e ".[dev]"


In [ ]:
#@title 2. Upload target FASTA

#@markdown Upload a FASTA containing the whole plasmid (one or more gene records).
#@markdown Headers can include `start:` and `end:` to mark the ORF.

from google.colab import files
import os

uploaded = files.upload()
target_file = next(iter(uploaded))
print(f"Uploaded: {target_file} ({len(uploaded[target_file])} bytes)")

!mkdir -p workspace
!mv "$target_file" workspace/
directory = "workspace/"


In [ ]:
#@title 3. Configure parameters and run DIMPLE
import ast
import logging
import os
from datetime import datetime

from Bio.Seq import Seq

from DIMPLE.DIMPLE import (
    addgene,
    align_genevariation,
    generate_DMS_fragments,
    post_qc,
    print_all,
)
from DIMPLE.run_settings import (
    DimpleRuntimeConfig,
    apply_barcode_start,
    apply_handle,
    apply_instance_settings,
    apply_random_seed,
    apply_restriction_settings,
    apply_runtime_policies,
    compute_overlaps_and_maxfrag,
    configure_dimple_logging,
    normalize_avoid_list,
    resolve_codon_usage,
    validate_insertions,
)
from DIMPLE.utilities import parse_custom_mutations

# Fall back to a bundled fixture if no upload happened (used by smoke tests).
if "target_file" not in locals():
    target_file = "Kir.fa"
    directory = "tests/data/"
    print(f"No upload found — using fixture: {directory}{target_file}")

work_dir = directory

#@markdown ### Oligo parameters
oligo_len = 230 #@param {type:"integer"}
fragment_len = "auto" #@param {type:"string"}
overlap = 4 #@param {type:"integer"}
barcode_start = 0 #@param {type:"integer"}

#@markdown ### Restriction enzymes
#@markdown Format: `RECOG(buffer)cut/extend`. Examples: BsmBI `CGTCTC(G)1/5`, BsaI `GGTCTC(G)1/5`.
restriction_sequence = "CGTCTC(G)1/5" #@param ["CGTCTC(G)1/5", "GGTCTC(G)1/5"] {allow-input: true}
sequences_to_avoid = "CGTCTC, GGTCTC" #@param {type:"string"}

#@markdown ### Codon usage
codon_usage_choice = "human" #@param ["human", "ecoli"]
custom_codon_usage = False #@param {type:"boolean"}

#@markdown ### Mutation types (pick one or more)
select_deep_mutation_scan = True #@param {type:"boolean"}
select_domain_insertion = False #@param {type:"boolean"}
select_insertions = False #@param {type:"boolean"}
insertions_csv = "GGC,GGCTCT,GGCTCTGGA" #@param {type:"string"}
select_deletions = False #@param {type:"boolean"}
deletions_csv = "3,6" #@param {type:"string"}

#@markdown ### Domain insertion (only used if domain insertion is selected)
handle = "AGCGGGAGACCGGGGTCTCTGAGC" #@param {type:"string"}

#@markdown ### Mutation options
amino_acid_substitutions = "Cys,Asp,Ser,Gln,Met,Asn,Pro,Lys,Thr,Phe,Ala,Gly,Ile,Leu,His,Arg,Trp,Val,Glu,Tyr" #@param {type:"string"}
include_stop_codons = False #@param {type:"boolean"}
include_synonymous_mutations = False #@param {type:"boolean"}
make_double_mutations = False #@param {type:"boolean"}
maximize_nucleotide_change = False #@param {type:"boolean"}

#@markdown ### Custom mutations (optional)
select_custom_mutations = False #@param {type:"boolean"}

#@markdown ### Advanced
match_sequences = False #@param {type:"boolean"}
random_seed = 0 #@param {type:"integer"}
orf_index = 0 #@param {type:"integer"}
#@markdown `orf_index=0` means auto. Use 1/2/3 if your FASTA has multiple ORFs and you want a specific one.
breaksite_change_policy = "warn" #@param ["warn", "error"]

# Optional uploads guarded by their toggles.
usage = codon_usage_choice
if custom_codon_usage:
    print("Upload a codon usage file. Example: DIMPLE/data/custom_codon_usage.txt")
    uploaded_usage = files.upload()
    usage_file = next(iter(uploaded_usage))
    with open(usage_file) as f:
        usage = ast.literal_eval(f.read().strip())

custom_mutations = None
if select_custom_mutations:
    print("Upload a custom mutations file (one `position:AA` per line, header on row 1).")
    uploaded_cm = files.upload()
    cm_file = next(iter(uploaded_cm))
    with open(cm_file) as f:
        custom_mutations = parse_custom_mutations(f.readlines()[1:])

insertions = [s.strip() for s in insertions_csv.split(",")] if select_insertions else False
deletions = [int(s) for s in deletions_csv.split(",")] if select_deletions else False
avoid_sequence = [s.strip() for s in sequences_to_avoid.split(",") if s.strip()]

# Logging — keep the run trace next to the outputs.
os.makedirs(work_dir, exist_ok=True)
log_file = os.path.join(work_dir, "DIMPLE-{:%Y-%m-%d-%s}.log".format(datetime.now()))
configure_dimple_logging(log_file)
logger = logging.getLogger(__name__)
logger.info("Notebook run started")

# Build the runtime config — same flow as run_dimple.py.
runtime_config = DimpleRuntimeConfig()
apply_handle(handle, config=runtime_config)

fragment_len_arg = 0 if fragment_len.strip().lower() in ("", "auto") else int(fragment_len)
overlap_l, overlap_r = compute_overlaps_and_maxfrag(
    oligo_len,
    fragment_len_arg,
    overlap,
    deletions,
    logger=logger,
    config=runtime_config,
)

apply_barcode_start(int(barcode_start), config=runtime_config)
apply_restriction_settings(restriction_sequence, config=runtime_config)
normalize_avoid_list(avoid_sequence, logger=logger, config=runtime_config)

# Colab is non-interactive by definition — never prompt.
apply_runtime_policies(
    dms=select_deep_mutation_scan,
    stop_codon=include_stop_codons,
    make_double=make_double_mutations,
    maximize_nucleotide_change=maximize_nucleotide_change,
    non_interactive=True,
    preferred_orf_index=orf_index if orf_index else None,
    link_policy="never",
    breaksite_change_policy=breaksite_change_policy,
    config=runtime_config,
)

apply_random_seed(random_seed if random_seed else None, config=runtime_config)
resolve_codon_usage(usage, config=runtime_config)

if not any([runtime_config.dms, select_domain_insertion, insertions, deletions]):
    raise ValueError(
        "No mutations selected. Enable at least one of: DMS, domain insertion, "
        "insertions, deletions."
    )

# Build the pool and apply per-gene settings.
pool = addgene(os.path.join(work_dir, target_file).strip(), runtime_config)
apply_instance_settings(
    pool,
    config=runtime_config,
    aminoacids=[a.strip() for a in amino_acid_substitutions.split(",") if a.strip()],
)

if insertions:
    validate_insertions(list(insertions), runtime_config)

if match_sequences:
    align_genevariation(pool)

logger.info("Generating DMS fragments")
generate_DMS_fragments(
    pool,
    overlap_l,
    overlap_r,
    include_synonymous_mutations,
    custom_mutations,
    runtime_config.dms,
    insertions,
    deletions,
    select_domain_insertion,
    work_dir,
    config=runtime_config,
)
post_qc(pool, config=runtime_config)
print_all(pool, work_dir, config=runtime_config)
logger.info("Notebook run finished")
print("Done — outputs are in:", work_dir)


In [ ]:
#@title 4. Download results as a zip
from google.colab import files

!zip -r dimple_results.zip $work_dir
files.download("dimple_results.zip")
